# Pandas and Spark: a distributed-execution textbook

Spark changes where data is stored and which operations trigger computation. It
should not change the estimand or the definition of a test. This chapter connects
the HypEx backend abstraction with the practical details that make distributed
analyses correct and affordable.

## Learning objectives

- distinguish driver, executors, partitions, actions and lazy transformations;
- construct equivalent pandas and Spark `Dataset` objects;
- explain which tests are exact from aggregates and which are histogram
  approximations;
- compare backend results using appropriate numerical expectations;
- manage caching, Python/Java versions and local Spark resources;
- recognise the driver–distributed boundary in matching and reporting.

**Requirements:** Java 8/11/17 and PySpark 3.5.1. The local example uses two cores
and small data so the statistical behaviour remains easy to inspect.

## The execution model in five ideas

1. The **driver** builds a logical plan and coordinates work.
2. **Executors** process partitions of data in parallel.
3. DataFrame transformations are usually **lazy**: they extend the plan
   rather than immediately compute every row.
4. An **action** such as `count` or `collect` requests a result and triggers
   jobs. Collecting a large table to the driver defeats the purpose of
   distributing it.
5. A **shuffle** redistributes rows across partitions, for example by group.
   It involves disk, network and scheduling, so partition counts matter.

HypEx uses aggregate-first implementations where possible: distributed jobs
compute means, variances, counts or histograms, and only small statistics
reach the driver. The final test statistic and report can then be computed
locally.

## 1. Start a reproducible local session

`PYARROW_IGNORE_TIMEZONE` is set before importing PySpark to keep
pandas-on-Spark behaviour consistent. Both Spark Python variables point to
the current kernel interpreter so workers cannot accidentally use another
Python version. Few shuffle partitions avoid the overhead of Spark's
default 200 partitions on a tiny tutorial dataset.

### Why environment setup belongs in a reproducible notebook

Spark starts a JVM, so `java` must be a real runtime rather than a launcher
placeholder. Python workers must also use an interpreter compatible with
the driver; a system `python3` from another environment is a common source
of `PYTHON_VERSION_MISMATCH` and missing-package errors.

`local[2]` runs two worker threads in one JVM. `spark.driver.memory=2g`
caps driver memory, while four shuffle partitions avoid creating hundreds
of tiny tasks. These are tutorial settings, not universal cluster defaults.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


def java_is_available():
    try:
        return (
            subprocess.run(
                ["java", "-version"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                check=False,
            ).returncode
            == 0
        )
    except OSError:
        return False


if not java_is_available():
    java_candidates = [
        Path("/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk/libexec/openjdk.jdk/Contents/Home"),
    ]
    java_home = next(
        (path for path in java_candidates if (path / "bin/java").exists()),
        None,
    )
    if java_home is None:
        raise RuntimeError(
            "Java 8/11/17 is required. Set JAVA_HOME or add java to PATH. "
            "On macOS with Homebrew: brew install openjdk@11."
        )
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

from hypex import AATest, ABTest, Matching
from hypex.comparators import StatsZTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum

# Configure the local JVM before Spark starts, while keeping error messages visible.
spark_files = TemporaryDirectory(prefix="hypex-spark-")
log_config = Path(spark_files.name) / "log4j2.properties"
log_config.write_text(
    "rootLogger.level = error\n"
    "rootLogger.appenderRef.console.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %p %c: %m%n\n",
    encoding="utf-8",
)
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypExAdvancedTutorial")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.extraJavaOptions", f"-Dlog4j.configurationFile={log_config.as_uri()}")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
# Spark matching checkpoints its predictions. Use a temporary local directory.
spark.sparkContext.setCheckpointDir(str(Path(spark_files.name) / "checkpoints"))
print("Spark:", spark.version)

Spark: 3.5.1


## 2. Construct both datasets from the same rows

The roles are identical. Only `backend` and `session` differ. `data`
exposes the underlying pandas or Spark DataFrame, while `raw_data` is the
pandas-on-Spark representation on the Spark backend; most user code should
stay with the backend-agnostic `Dataset` API.

In [3]:
rng = np.random.default_rng(77)
n_users = 2_000
pre_revenue = rng.gamma(4, 20, n_users)
treat = rng.binomial(1, 0.5, n_users)
revenue = pre_revenue + 6 * treat + rng.normal(0, 12, n_users)
converted = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.02 * revenue))))
channel = np.where(
    treat == 1,
    rng.choice(["organic", "paid"], n_users, p=[0.6, 0.4]),
    rng.choice(["organic", "paid"], n_users, p=[0.5, 0.5]),
)
backend_df = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "treat": treat,
        "revenue": revenue,
        "converted": converted,
        "channel": channel,
        "age": rng.normal(38, 10, n_users),
        "pre_spend": pre_revenue,
    }
)
roles = {
    "user_id": InfoRole(int),
    "treat": TreatmentRole(int),
    "revenue": TargetRole(float),
    "converted": TargetRole(int),
    "channel": TargetRole(str),
    "age": FeatureRole(float),
    "pre_spend": FeatureRole(float),
}
pandas_data = Dataset(roles=roles, data=backend_df)
spark_data = Dataset(
    roles=roles,
    data=backend_df,
    backend=BackendsEnum.spark,
    session=spark,
)
print("pandas:", type(pandas_data.data).__module__)
print("Spark:", type(spark_data.data).__module__)
print("Backend enum:", spark_data.backend_type)

pandas: pandas.core.frame
Spark: pyspark.sql.dataframe
Backend enum: BackendsEnum.spark


### What changes between the two constructors

Roles, column names and row order are the same. The Spark constructor
additionally receives a session and distributes the DataFrame. Statistical
code should therefore depend on `Dataset`, not on pandas-only methods.

Check the number of partitions as an execution detail. More partitions do
not automatically mean faster execution: for 2,000 rows, many partitions
would create mostly empty tasks.

In [4]:
print("Spark partitions:", spark_data.data.rdd.getNumPartitions())
print("Rows (action):", spark_data.data.count())

Spark partitions: 2


Rows (action): 2000


## 3. Run the same A/B pipeline

On pandas, t-, KS- and U-tests operate on raw columns through SciPy. On
Spark, their registered implementations aggregate first. T-test moments and
chi-square counts support equivalent calculations; KS and Mann–Whitney use
histogram approximations, so small numerical differences are expected.

### How a t-test works without collecting raw columns

For group $g$, Spark computes $n_g$, $\bar Y_g$ and $s_g^2$. These are
sufficient for the Welch statistic:

$$T=\frac{\bar Y_1-\bar Y_0}
{\sqrt{s_1^2/n_1+s_0^2/n_0}}.$$

Means and counts can be combined exactly across partitions. Sample
variance uses sums $\sum Y$ and $\sum Y^2$ or stable parallel variants.
Floating-point accumulation order can still produce tiny numerical
differences, so “exact” refers to the statistic's definition, not bitwise
identity.

Chi-square similarly needs category counts. In contrast, exact KS and
Mann–Whitney statistics require order information that is less natural to
express as a handful of moments. Spark uses shared histogram summaries for
these approximations.

In [5]:
def make_ab_test():
    return ABTest(
        additional_tests=["t-test", "ks-test", "u-test", "chi2-test"],
        multitest_method="holm",
    )


pandas_result = make_ab_test().execute(pandas_data)
spark_result = make_ab_test().execute(spark_data)
spark_result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value,UTest pass,UTest p-value,Chi2Test pass,Chi2Test p-value
0,revenue,1,79.388506,86.256069,6.867563,8.650576,3.266323,10.468802,OK,0.000191,OK,0.000064,OK,0.000030,NaN,NaN
1,converted,1,0.331373,0.323469,-0.007903,-2.384978,-0.049062,0.033255,NOT OK,0.706692,NOT OK,1.000000,NOT OK,0.706662,NaN,NaN
2,channel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,5.889447e-10


In [6]:
comparison_columns = [
    "feature",
    "group",
    "difference",
    "TTest p-value",
    "KSTest p-value",
    "UTest p-value",
    "Chi2Test p-value",
]
pandas_summary = pandas_result.summary.raw_data[comparison_columns]
spark_summary = spark_result.summary.raw_data[comparison_columns]
parity = pandas_summary.merge(
    spark_summary,
    on=["feature", "group"],
    suffixes=(" pandas", " spark"),
)
parity.round(5)

,feature,group,difference pandas,TTest p-value pandas,KSTest p-value pandas,UTest p-value pandas,Chi2Test p-value pandas,difference spark,TTest p-value spark,KSTest p-value spark,UTest p-value spark,Chi2Test p-value spark
0,revenue,1,6.86756,0.00019,0.00005,0.00003,NaN,6.86756,0.00019,0.00006,0.00003,NaN
1,converted,1,-0.00790,0.70669,1.00000,0.70666,NaN,-0.00790,0.70669,1.00000,0.70666,NaN
2,channel,1,NaN,NaN,NaN,NaN,0.00001,NaN,NaN,NaN,NaN,0.0


### A parity check needs a tolerance and a reason

For `difference` and t-test p-values, compare within floating-point
tolerance. For KS and U, inspect the approximation error and whether the
inferential conclusion is stable. Do not hide a large discrepancy by
choosing an arbitrary loose tolerance.

The approximate KS statistic uses empirical CDFs evaluated at shared bin
edges. If $B$ is the ordered set of edges and $H_g(b)$ is the cumulative
count for group $g$, then approximately

$$D\approx\max_{b\in B}
\left|\frac{H_1(b)}{n_1}-\frac{H_0(b)}{n_0}\right|.$$

An extreme outlier widens the global range and makes most bins coarser.
Increasing `n_bins` can help resolution but does not remove the underlying
approximation or its computational cost.

The effect estimates should agree up to floating-point aggregation order.
For approximate tests, assess the conclusion and the magnitude of the
difference rather than demanding bit-for-bit equality. Histogram-based KS
and U tests use `n_bins`; extreme outliers can stretch the bins and reduce
approximation quality.

## 4. Aggregate-first statistics as standalone calculators

`StatsZTest` needs only counts and sums to compare two proportions:

$$z=\frac{\hat p_1-\hat p_2}
{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}.$$

On Spark this means a distributed aggregation followed by a small driver-
side calculation, not a transfer of every raw outcome to SciPy.

In [7]:
ztest = StatsZTest.calc(
    target_fields_data=spark_data[["converted"]],
    group_field_data=spark_data[["treat"]],
    reliability=0.05,
)
ztest[0].raw_data

,p-value,statistic,pass
0,0.706597,0.376485,0.0


### Read the proportion test with its assumptions

Here $x_g$ is the number of conversions, $n_g$ is users, and
$\hat p_g=x_g/n_g$. Under equal proportions, the pooled estimate is
$\hat p=(x_1+x_0)/(n_1+n_0)$.

The normal approximation is most reliable when expected successes and
failures are not tiny. Rare conversion, small groups or complete
separation can make the z approximation uninformative; an exact or
regression-based approach may then be more appropriate.

`StatsZTest.calc` returns a list of result tables on this version. The
statistical calculation is small; the distributed aggregation that produced
its inputs is the important part.

## 5. Cache deliberately, then release

Repeated splits and aggregations can otherwise recompute the same Spark
lineage. Shells auto-persist an uncached Spark input for the duration of
`execute`, then release only what they cached themselves. A dataset you
explicitly persist remains cached after the shell returns.

### Persistence controls recomputation, not statistical correctness

`MEMORY_AND_DISK` keeps partitions in memory where possible and spills to
disk when needed. It is useful when the same lineage is scanned repeatedly.
`MEMORY_ONLY` avoids that spill path but may recompute evicted partitions;
`DISK_ONLY` trades memory pressure for more I/O.

`persist(..., action="count")` both caches and materialises the dataset.
Without an action, caching is lazy and the first later action pays the
computation cost.

HypEx shells track ownership: they unpersist only a dataset they persisted
themselves. Your manually cached input remains yours to release. This
avoids both uncontrolled cache growth and accidentally invalidating a
caller's reuse plan.

In [8]:
spark_data.persist(storage_level="MEMORY_AND_DISK", action="count")
print(
    {
        "is_persisted": spark_data.is_persisted,
        "storage_level": spark_data.get_storage_level(),
    }
)
cached_result = ABTest(additional_tests="t-test").execute(spark_data)
print("Still cached after shell execution:", spark_data.is_persisted)
spark_data.unpersist()
print("After explicit release:", spark_data.is_persisted)

{'is_persisted': True, 'storage_level': 'MEMORY_AND_DISK'}


Still cached after shell execution: True
After explicit release: False


## 6. A/A and matching also keep the same shell API

The A/A example uses only ten iterations to keep the tutorial quick. Use
hundreds or thousands for a serious false-positive-rate study. Matching
still uses FAISS for nearest-neighbour search: Spark distributes the data
preparation, but this is not a promise that every model operation becomes
distributed. Wide `full_data` extraction is left off because joins and
checkpoints are expensive.

In [9]:
spark_aa = AATest(n_iterations=10, stratification=False).execute(spark_data)
spark_aa.aa_score

  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:01<00:14,  1.56s/it]

 20%|██        | 2/10 [00:02<00:11,  1.44s/it]

 30%|███       | 3/10 [00:04<00:09,  1.34s/it]

 40%|████      | 4/10 [00:05<00:07,  1.27s/it]

 50%|█████     | 5/10 [00:06<00:06,  1.23s/it]

... (3 outputs left out)

 90%|█████████ | 9/10 [00:10<00:01,  1.08s/it]

100%|██████████| 10/10 [00:11<00:00,  1.06s/it]

100%|██████████| 10/10 [00:11<00:00,  1.17s/it]

,score,pass
revenue TTest test_1,0.95,True
converted TTest test_1,0.95,True
age TTest test_1,0.95,True
pre_spend TTest test_1,0.95,True
revenue KSTest test_1,0.95,True
converted KSTest test_1,0.95,True
age KSTest test_1,0.95,True
pre_spend KSTest test_1,0.95,True
channel Chi2Test test_1,0.95,True


### Why repeated experiments amplify Spark overhead

Every A/A iteration contains assignment, aggregations and reporting. Ten
iterations are enough to demonstrate the API, not to estimate FPR. A
serious Monte Carlo study can trigger many jobs, so reuse a clean persisted
dataset and avoid reconstructing sessions or reloading source data inside
the loop.

Spark does not remove Monte Carlo uncertainty. The same binomial logic and
p-value diagnostics from Chapter 01 apply.

In [10]:
matching_roles = {
    "user_id": InfoRole(int),
    "treat": TreatmentRole(int),
    "revenue": TargetRole(float),
    "age": FeatureRole(float),
    "pre_spend": FeatureRole(float),
    "channel": FeatureRole(str),
}
matching_columns = [
    "user_id",
    "treat",
    "revenue",
    "age",
    "pre_spend",
    "channel",
]
spark_matching_data = Dataset(
    roles=matching_roles,
    data=backend_df[matching_columns],
    backend=BackendsEnum.spark,
    session=spark,
)
spark_matching = Matching(
    distance="mahalanobis",
    quality_tests=[],
    extract_full_data=False,
).execute(spark_matching_data)
spark_matching.summary

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,7.1095,0.9596,0.0,5.2286,8.9904
ATC,7.0903,0.9444,0.0,5.2393,8.9414
ATE,7.0997,0.8537,0.0,5.4265,8.7730


### Matching is a hybrid workload

Spark can prepare features, join indexes and run aggregate quality checks.
FAISS nearest-neighbour search has its own in-memory computational model,
so “Spark backend” does not mean that every neighbour search is partitioned
across the cluster.

`full_data` performs iterative joins and can expand Spark execution plans.
It is disabled here deliberately. Request it when the matched table itself
is needed, and budget for checkpoints, storage and join complexity.

Stop the session at the end of a local notebook to release JVM and Python
worker resources. In a shared cluster, follow the platform's lifecycle
rules instead of stopping another user's session.

`Matching.execute` cleans up temporary FAISS index files automatically. If you call the Spark FAISS extension directly, see [explicit cleanup with FaissIndexStorage](../configuration/10_Spark_Matching_Configuration.ipynb#index-cleanup).

In [11]:
try:
    spark.stop()
finally:
    spark_files.cleanup()

## Backend checklist

- Keep roles and experiment definitions backend-independent.
- Expect exact parity for moment-based tests within numerical tolerance,
  and approximate parity for histogram-based KS/U tests.
- Reduce `spark.sql.shuffle.partitions` for small local jobs; tune it for
  data size on a cluster.
- Persist data reused across stages, but unpersist it when finished.
- Remember that result tables are intentionally small pandas-backed
  `SmallDataset` objects even when raw data stays distributed.
- Test the exact executor combination on Spark before scaling up; optional
  model and reporting operations have different cost profiles.

Return to the [advanced tutorials index](README.md).

## A production troubleshooting map

| Symptom | Likely execution issue | What to check |
|---|---|---|
| JVM exits before opening a port | Java unavailable or incompatible | Actual `java -version`, `JAVA_HOME` |
| Worker cannot import a package | Different Python environment | `PYSPARK_PYTHON`, worker environment |
| Python minor-version mismatch | Driver and workers use different interpreters | Both Spark Python variables |
| Many tiny, slow jobs | Too many shuffle partitions | Data size and partition settings |
| Driver runs out of memory | Large collect or wide report | Aggregations, matched-data extraction |
| Repeated lineage computation | Input not persisted across actions | Cache ownership and storage level |
| KS/U differs from pandas | Histogram approximation | `n_bins`, outliers, tolerance |
| Matching becomes very expensive | FAISS boundary plus iterative joins | Feature count, neighbours, checkpoints |

The key principle is to optimise **data movement and recomputation** while
preserving the statistical contract. Faster execution with a different
estimand is not a successful migration.